# Retrieval augmented generation demo

This tutorial first 'encodes' ROCStories in a memory store (this corresponds to the model hippocampus in our paper).

It then demonstrates retrieval augmented generation. Given a query, it first retrieves relevant stories, then prompts Mistral 7B with the retrieved context. 

See notebook 4 for the compression element, which isn't simulated here.

(You need to run the Mistral generation cells on a large GPU.)

## Build external memory

`LocalEmbedder` wraps a sentence-transformers model. The first run may download the embedding model. Each memory document is a full ROCStory.

In [1]:
from llm_psychology.data.rocstories import ROC_SENTENCE_COLUMNS, load_rocstories
from llm_psychology.rag.embed import LocalEmbedder
from llm_psychology.rag.memory import VectorStore
from llm_psychology.rag.retrieve import Retriever

stories = load_rocstories(limit=50, shuffle=True, seed=9)
documents = stories["text"].astype(str).tolist()

embedder = LocalEmbedder()
dim = embedder.encode(["dimension probe"]).shape[1]
store = VectorStore(dim=dim)
retriever = Retriever(embedder, store)
retriever.index(documents)
documents[:3]

/nfs/nhome/live/espens/.pyenv/versions/3.11.5/envs/my_venv_3115/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['Lynn gets a new dog. She loves her dog. She is out one evening walking him. Her dog runs off away from her. She cannot catch him and he runs away.',
 "Mildred drove her family everywhere when they traveled. On this occasion she was driving along a winding mountain road. A driver behind Mildred was tailgating and making her mad. She contemplated breaking suddenly to scare them off. Instead Mildred pulled over and let them pass for her family's safety.",
 'Tom was dating a girl. His best friend seemed to like her too. One day Tom found them together. He was upset with both. It ruined the friendship and relationship.']

## Inspect retrieval

Given a query, the first stage of RAG is to reteieve relevant memories from the store. This corresponds to hippocampal retrieval of relevant episodes into working memory.

Before prompting Mistral, inspect the retrieved ROCStories:

In [2]:
def paper_recall_question(row):
    return f"{row['sentence1']} What happened (in detail)?"


def paper_recall_target(row):
    return " ".join(str(row[column]) for column in ROC_SENTENCE_COLUMNS[1:])


example_rows = [row for _, row in stories.head(3).iterrows()]
questions = [paper_recall_question(row) for row in example_rows]
targets = [paper_recall_target(row) for row in example_rows]

retrieved = retriever.retrieve(questions, top_k=3)
for question, target, results in zip(questions, targets, retrieved):
    print("\nQ:", question)
    print("Target:", target)
    for score, text in results:
        print(f"  {score:.3f}  {text}")


Q: Lynn gets a new dog. What happened (in detail)?
Target: She loves her dog. She is out one evening walking him. Her dog runs off away from her. She cannot catch him and he runs away.
  0.703  Lynn gets a new dog. She loves her dog. She is out one evening walking him. Her dog runs off away from her. She cannot catch him and he runs away.
  0.475  Tom was on his way back to work. He didn't have enough time to stop to eat. He decided to get a street hot dog. Tom was disgusted by the quality and taste. He vowed never to eat street food again.
  0.405  Mindy had a little boy she loved. When she got divorced her husband got custody of their son. Mindy was very lonely. She decided to have another baby. She recently had a baby daughter that she loves.

Q: Mildred drove her family everywhere when they traveled. What happened (in detail)?
Target: On this occasion she was driving along a winding mountain road. A driver behind Mildred was tailgating and making her mad. She contemplated breaking

## Load Mistral

The default is `mistralai/Mistral-7B-Instruct-v0.2`. Override with `RAG_MISTRAL_MODEL` for a local path or a different Mistral checkpoint.

In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = os.environ.get("RAG_MISTRAL_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")
model_kwargs = {}
if torch.cuda.is_available():
    model_kwargs.update({"torch_dtype": torch.bfloat16, "device_map": "auto"})

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(model_name)
print(next(model.parameters()).device)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████| 3/3 [00:41<00:00, 13.88s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


mistralai/Mistral-7B-Instruct-v0.2
cuda:0


## Prompt Mistral With retrieved context

The generation part of RAG is simply prompting the LLM with the retrieved 'memories'.

The prompt follows the paper's Mistral instruction format for recalled ROCStories, with the retrieved full story placed under `Background:`.

In [4]:
def retrieve_background(question, top_k=1):
    retrieved_docs = [text for _score, text in retriever.retrieve([question], top_k=top_k)[0]]
    if top_k == 1:
        return retrieved_docs[0]
    return "\n\n".join(
        f"Background document {index}: {text}"
        for index, text in enumerate(retrieved_docs, start=1)
    )


def build_mistral_rag_prompt(question, top_k=1):
    background = retrieve_background(question, top_k=top_k)
    return (
        "[INST] Refer to the background document and answer the question.\n"
        f"Background: {background}\n"
        f"Question: {question} [/INST] The answer is:"
    )


prompt = build_mistral_rag_prompt(questions[0])
print(prompt)

[INST] Refer to the background document and answer the question.
Background: Lynn gets a new dog. She loves her dog. She is out one evening walking him. Her dog runs off away from her. She cannot catch him and he runs away.
Question: Lynn gets a new dog. What happened (in detail)? [/INST] The answer is:


In [5]:
device = next(model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=int(os.environ.get("RAG_MISTRAL_MAX_NEW_TOKENS", "96")),
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
answer_text = tokenizer.decode(generated[0], skip_special_tokens=True)
print(answer_text)

[INST] Refer to the background document and answer the question.
Background: Lynn gets a new dog. She loves her dog. She is out one evening walking him. Her dog runs off away from her. She cannot catch him and he runs away.
Question: Lynn gets a new dog. What happened (in detail)? [/INST] The answer is: Lynn recently acquired a new dog and developed a strong attachment to him. One evening, as they were out for a walk, Lynn was enjoying the company of her beloved pet. However, suddenly and unexpectedly, her dog got spooked or distracted by something in the environment and broke free from her leash or grip. Lynn tried her best to catch up with her dog, but he was too quick and managed to run away from her. Despite her efforts
